# Dự án 6 — Dự báo và đưa model ra đời thật — Đề B: Không khí TP.HCM — bụi mịn ngày mai
## BẢN CỦA NHÓM

**Trường THCS và THPT Đinh Thiện Lý — Machine Learning Level 1 — Khối 10**

---

Bụi mịn PM2.5 ảnh hưởng sức khoẻ. Nếu biết trước ngày mai bụi nhiều, trường có thể dời giờ chơi ngoài trời. Bảng có **4 năm, mỗi ngày một dòng** (2022 – 2026) cho TP.HCM.
Đề này nối với Bài 30 (bụi mịn Bắc Kinh) — nhưng dữ liệu Việt Nam có một điều bất ngờ. Nhóm làm thám tử.

**Câu hỏi lớn của dự án:** *Ngày mai thế nào? Và người khác dùng model của nhóm ra sao?*

**Dữ liệu:** Kaggle — HCMC Air Quality 2022 – 2026 (gốc OpenWeatherMap), **CC BY-NC-SA 4.0**  

> Số liệu là **ước tính theo mô hình** của OpenWeatherMap, không phải trạm đo. Không dùng để ra quyết định sức khoẻ.

### Các mốc (nộp trên Canvas trước 23:59 Thứ 7 của tuần đó)

| Tuần | Mốc | Nộp |
|---|---|---|
| 1 · buổi 25 | **M1 + M2** Chuỗi thời gian, chia theo thời gian, mốc “mai như hôm nay” | Notebook bước 1 – 2 |
| 2 · buổi 26 | **M3** Hồi quy và mạng nơ-ron nhỏ so với mốc; 10 ngày sai nhiều nhất | Notebook bước 3 |
| 3 · buổi 27 | **M4a** App Gradio, chặn đầu vào vô lý | Notebook bước 4 + link app |
| 4 · buổi 28 | **M4b** Nhóm bạn dùng thử, ghi lỗi; nhóm sửa; báo cáo | Notebook đủ + bảng thử + slide |

### Nhóm và vai — **đổi vai mỗi tuần**

Nhóm: … · Lớp: …

| Vai | Tuần 1 | Tuần 2 | Tuần 3 | Tuần 4 | Việc chính |
|---|---|---|---|---|---|
| Lái code | … | … | … | … | Gõ và chạy notebook; không ai gõ hộ hai tuần liền |
| Nghiên cứu | … | … | … | … | Đọc bài thư viện, hỏi Gemini “vì sao”, ghi Nhật ký AI |
| Thư ký số liệu | … | … | … | … | Ghi bảng thí nghiệm, kiểm tra con số trước khi báo cáo |
| Trình bày | … | … | … | … | Báo cáo 1 phút mỗi tuần, làm slide / poster |

### Quy tắc dùng AI

- **Được:** hỏi Gemini (✨ trong Colab) một lệnh làm gì, bấm *Giải thích lỗi*, hỏi *vì sao* làm bước này, nhờ gợi ý cách tiếp cận.
- **Không được:** nộp con số, biểu đồ, kết luận mà nhóm không tự chạy ra; để AI viết cả notebook.
- Ô code do AI gợi ý ghi dòng đầu `# AI gợi ý — nhóm đã kiểm tra: ...`. Mỗi tuần ghi **Nhật ký AI** ở cuối.
- Khi đi qua nhóm và ở mốc cuối, thầy gọi **một bạn bất kỳ** giải thích một dòng code hoặc một con số.

### Dữ liệu

| Cột | Ý nghĩa |
|---|---|
| `ngay` | Ngày |
| `pm25, pm10` | Bụi mịn trung bình trong ngày (µg/m³) — **pm25 là chuỗi cần dự báo** |
| `nhiet_do, do_am, gio` | Nhiệt độ (°C), độ ẩm (%), gió (km/h) trung bình |
| `mua` | Tổng mưa (mm) |


---
# TUẦN 1 (buổi 25) — Chuỗi thời gian và mốc

**Mốc của tuần:** **M1 + M2** Chuỗi thời gian, chia theo thời gian, mốc “mai như hôm nay”  
**Học trước (bài thư viện trên web, 3 chặng bắt buộc + checkpoint):** [Bài 30 Dự báo theo thời gian](https://vuongndlst.github.io/ml-level1/bai30/)


---
## Bước 0 — Nạp dữ liệu (đã viết sẵn, chỉ cần chạy)


In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import mean_absolute_error
URL = "https://vuongndlst.github.io/ml-level1/du-lieu/"
def doc(tep, **kw):   # đã tải file lên Colab thì đọc file; chưa thì đọc thẳng từ web khoá học
    return pd.read_csv(tep if os.path.exists(tep) else URL + tep, **kw)
def tao_cua_so(s, so_ngay=7):
    """Như Bài 30: biến chuỗi s thành bảng cột t-1 … t-so_ngay (các ngày trước) -> y (ngày cần đoán)."""
    bang = pd.DataFrame({"t-%d" % i: s.shift(i) for i in range(1, so_ngay + 1)})
    bang["y"] = s
    return bang.dropna()
CAT = "2025-04-01"            # ngày bắt đầu tập kiểm tra
DON_VI = "µg/m³"
TEN_APP = "Dự báo bụi mịn TP.HCM"
THU_APP = [(15, 12, 10), (500, 20, 20), (-3, 10, 9)]   # 3 lần thử app: (hôm nay, hôm qua, hôm kia)
KHONG_AM = 0            # dự báo không được nhỏ hơn số này
df = doc("khong_khi_hcm_ngay.csv", parse_dates=["ngay"], index_col="ngay")
print(df.shape)
df.head()


### Câu hỏi của nhóm (M1)

*Gợi ý câu hỏi:* Ngày mai bụi bao nhiêu? · Tháng nào bụi nhất? · Model có thắng “mai như hôm nay”?

| | Nhóm điền |
|---|---|
| Câu hỏi nhóm chọn | … |
| Cột sẽ dùng | … |
| Dự đoán trước (giả thuyết) | … |
| Làm sao biết dự đoán đúng hay sai? | … |


---
## Bước 1 — Nhìn chuỗi thời gian

> **🎯 Mục tiêu:** Vẽ chuỗi, tìm ngày trống, mùa, xu hướng, điều bất thường.
>
> **🧭 Vì sao bước này?** Chuỗi thời gian có thứ tự: hôm qua ảnh hưởng hôm nay. Nhìn trước để biết mùa, xu hướng, chỗ lạ.
>
> **📋 Nhiệm vụ:**
> 1. Tạo chuỗi `s` (mỗi ngày một số).
> 2. Vẽ chuỗi theo thời gian.
> 3. Tính trung bình theo tháng / năm.
>
> **💡 Gợi ý lệnh:**
> - `s.plot()` — vẽ chuỗi
> - `s.groupby(s.index.month).mean()` — theo tháng
>
> **✅ Đạt khi:** có biểu đồ chuỗi và một nhận xét về mùa / xu hướng.


In [ ]:
# Buoc 1 - code cua nhom



**Nhận xét của nhóm:** …


---
## Bước 2 — Chia theo thời gian và mốc

> **🎯 Mục tiêu:** Biến chuỗi thành bảng 7 ngày trước → ngày cần đoán; chia theo ngày; tính 3 mốc.
>
> **🧭 Vì sao bước này?** Xáo trộn rồi chia là dùng tương lai để đoán quá khứ — con số đẹp giả.
>
> **📋 Nhiệm vụ:**
> 1. `tao_cua_so(s, 7)`.
> 2. Tập kiểm tra = các ngày từ `CAT` trở đi — **không xáo trộn**.
> 3. Ba mốc: mai như hôm nay, như tuần trước, luôn đoán trung bình.
>
> **💡 Gợi ý lệnh:**
> - `tao_cua_so(s, 7)` — bảng cửa sổ trượt
> - `bang[bang.index < CAT]` — chia theo thời gian
>
> **✅ Đạt khi:** có số ngày học / kiểm tra và MAE của 3 mốc.


In [ ]:
# Buoc 2 - code cua nhom



**Nhận xét của nhóm:** …


### ✅ Nộp M1 + M2 (tuần 1)

- [ ] Bảng câu hỏi của nhóm
- [ ] Bước 1 – 2 có nhận xét kèm số
- [ ] *Runtime → Restart and run all* chạy hết không lỗi
- [ ] Link notebook (*Chia sẻ → Bất kỳ ai có đường liên kết · Người xem*) nộp lên Canvas trước 23:59 Thứ 7


---
# TUẦN 2 (buổi 26) — Hồi quy, mạng nơ-ron và ngày sai

**Mốc của tuần:** **M3** Hồi quy và mạng nơ-ron nhỏ so với mốc; 10 ngày sai nhiều nhất  
**Học trước (bài thư viện trên web, 3 chặng bắt buộc + checkpoint):** [Bài 29 Mạng nơ-ron](https://vuongndlst.github.io/ml-level1/bai29/)


---
## Bước 3 — Model so với mốc

> **🎯 Mục tiêu:** Hồi quy tuyến tính và mạng nơ-ron nhỏ trên 7 ngày trước; so mốc; 10 ngày sai nhiều nhất.
>
> **🧭 Vì sao bước này?** Model phức tạp chưa chắc thắng mốc đơn giản. Phải so, và phải thật thà khi thua.
>
> **📋 Nhiệm vụ:**
> 1. Huấn luyện hai model trên tập học.
> 2. MAE trên tập kiểm tra, so với mốc.
> 3. Đoán lệch có dấu: model hay đoán cao hay thấp?
> 4. Xem 10 ngày sai nhiều nhất.
>
> **💡 Gợi ý lệnh:**
> - `MLPRegressor(hidden_layer_sizes=(16,), max_iter=2000, random_state=0)` — mạng nơ-ron 16 nơ-ron, như Bài 29
>
> **✅ Đạt khi:** có MAE ba cách, biểu đồ 60 ngày, 10 ngày sai và một nhận xét.


In [ ]:
# Buoc 3 - code cua nhom



**Nhận xét của nhóm:** …


### ✅ Nộp M3 (tuần 2)

- [ ] Bước 3: MAE ba cách + nhận xét về 10 ngày sai
- [ ] *Runtime → Restart and run all* chạy hết không lỗi
- [ ] Link notebook (*Chia sẻ → Bất kỳ ai có đường liên kết · Người xem*) nộp lên Canvas trước 23:59 Thứ 7


---
# TUẦN 3 (buổi 27) — App Gradio

**Mốc của tuần:** **M4a** App Gradio, chặn đầu vào vô lý  
**Học trước (bài thư viện trên web, 3 chặng bắt buộc + checkpoint):** [Bài 31 Đưa model thành ứng dụng](https://vuongndlst.github.io/ml-level1/bai31/)


---
## Bước 4 — App cho người khác dùng

> **🎯 Mục tiêu:** Hàm dự báo có chặn đầu vào vô lý; app Gradio 3 ô nhập.
>
> **🧭 Vì sao bước này?** Model trong notebook chưa giúp được ai. App biến model thành công cụ — nhưng phải chống dùng sai.
>
> **📋 Nhiệm vụ:**
> 1. Model 3 ngày gần nhất (dễ nhập).
> 2. Hàm `du_doan_an_toan` chặn số ngoài vùng dữ liệu, ô trống.
> 3. Thử hàm với 3 đầu vào, rồi mở app trên Colab, gửi link cho nhóm bạn.
>
> **💡 Gợi ý lệnh:**
> - `gr.Interface(ham, [gr.Number(...)] * 3, gr.Textbox())` — app 3 ô nhập
> - `app.launch(share=True)` — link dùng thử tạm thời
>
> **✅ Đạt khi:** có hàm chặn đầu vào và link app.


In [ ]:
# Buoc 4 - code cua nhom



**Nhận xét của nhóm:** …


### ✅ Nộp M4a (tuần 3)

- [ ] Bước 4 chạy được; link app dán vào Canvas
- [ ] *Runtime → Restart and run all* chạy hết không lỗi
- [ ] Link notebook (*Chia sẻ → Bất kỳ ai có đường liên kết · Người xem*) nộp lên Canvas trước 23:59 Thứ 7


---
# TUẦN 4 (buổi 28) — Nhóm bạn dùng thử, nhóm sửa

**Mốc của tuần:** **M4b** Nhóm bạn dùng thử, ghi lỗi; nhóm sửa; báo cáo  
**Học trước (bài thư viện trên web, 3 chặng bắt buộc + checkpoint):** —


---
## Bước 5 — Nhóm bạn dùng thử, nhóm sửa

> **🎯 Mục tiêu:** Nhóm bạn thử app ít nhất 6 kiểu đầu vào; nhóm ghi lỗi, sửa, thử lại.
>
> **🧭 Vì sao bước này?** Người dùng thật luôn nhập thứ mình không ngờ. Cho người khác thử là cách nhanh nhất để thấy lỗi.
>
> **📋 Nhiệm vụ:**
> 1. Nhóm bạn điền bảng thử (kiểu thử, đầu vào, app trả lời).
> 2. Nhóm chọn 1 – 2 lỗi để sửa.
> 3. Thử lại sau khi sửa.
>
> **💡 Gợi ý lệnh:**
> - `pd.DataFrame([...], columns=[...])` — bảng thử
>
> **✅ Đạt khi:** có bảng thử trước / sau khi sửa.


In [ ]:
# Buoc 5 - code cua nhom



**Nhận xét của nhóm:** …


### Slide kết quả (M4) — một slide, 4 ô

| Ô | Nội dung |
|---|---|
| Câu hỏi | … |
| Một biểu đồ | … |
| Một con số | … |
| App | … |
| Giới hạn | … |

*Làm trên Canva hoặc Google Slides; dán link vào ô nộp Canvas cùng link notebook.*


---
## Bước 6 — Nhật ký AI

Không dùng AI thì ghi *Không dùng*. Kiểm chứng = nhóm tự chạy lại, đối chiếu số, sửa chỗ AI sai.

| Nhóm hỏi AI | AI trả lời / viết | Nhóm kiểm chứng và sửa |
|---|---|---|
| … | … | … |
| … | … | … |
| … | … | … |


---
## Mở rộng — nhóm xong sớm (không bắt buộc)

1. Chỉ học trên 1 năm gần nhất trước tập kiểm tra — model còn đoán quá cao không?
2. Dự báo **mức thay đổi** (`s.diff()`) thay vì giá trị — có tránh được việc trôi dữ liệu không?


In [ ]:
# Mo rong - code cua nhom



---
## Trước khi nộp

- [ ] *Runtime → Restart and run all* chạy hết không lỗi
- [ ] Mỗi bước có một ô nhận xét **có số và cỡ nhóm**
- [ ] Không có câu “… **gây ra** …” khi dữ liệu chỉ cho thấy đi cùng nhau
- [ ] Bảng vai trò và nhật ký AI đã điền
- [ ] Link notebook ở chế độ *Người xem*, dán lên Canvas cùng link Canva
